# 15. MCP Client Development

This notebook demonstrates how to use the **FastMCP 3** client against the
Atlas Curated MCP Servers endpoint, covering discovery, invocation, error
handling, and in-process experimental tool authoring.

The MCP endpoint is available when `MCP_SERVERS_SOURCE=container` (disabled
by default). When disabled, the cells below handle the unavailable-state
gracefully.

In [ ]:
import os

from fastmcp import Client

MCP_SERVERS_URL = os.environ.get("MCP_SERVERS_URL", "")
if not MCP_SERVERS_URL:
    print("⚠ MCP_SERVERS_URL is not set — MCP_SERVERS_SOURCE is likely disabled.")
    print("  Enable with: ./start.sh --mcp-servers-source container")
else:
    print(f"MCP endpoint: {MCP_SERVERS_URL}")

## Discovery

List the tools the curated MCP server exposes, including their input schemas.

In [ ]:
if MCP_SERVERS_URL:
    async with Client(MCP_SERVERS_URL) as client:
        tools = await client.list_tools()
        for tool in tools:
            print(f"  {tool.name}: {tool.description or '(no description)'}")
            if tool.inputSchema:
                print(f"    schema: {tool.inputSchema}")
else:
    print("(MCP disabled — skipping discovery)")

## Invocation

Call a bounded read-only tool with safe fixture inputs. Replace `tool_name`
with a real tool from the discovery output above.

In [ ]:
if MCP_SERVERS_URL:
    async with Client(MCP_SERVERS_URL) as client:
        tools = await client.list_tools()
        if not tools:
            print("(no tools registered on this MCP endpoint)")
        else:
            # Call the first discovered tool, filling required string
            # arguments from its own input schema.
            tool = tools[0]
            schema = tool.inputSchema or {}
            required = schema.get("required", [])
            args = {name: "Atlas" for name in required}
            result = await client.call_tool(tool.name, args)
            print(f"{tool.name}({args}) -> {result}")
else:
    print("(MCP disabled — skipping invocation)")


## Error Handling

Handle the common failure modes: disabled service, authentication, timeout,
and tool errors.

In [ ]:
if MCP_SERVERS_URL:
    try:
        async with Client(MCP_SERVERS_URL) as client:
            result = await client.call_tool("nonexistent_tool", {})
    except Exception as exc:
        print(f"Caught expected error: {type(exc).__name__}: {exc}")
        print("This is normal — the tool does not exist.")
else:
    print("(MCP disabled — skipping error handling)")

## In-Process Experimental Tools

FastMCP lets you define and test a tool **in-process** — no network service
needed. This is useful for prototyping new MCP tools before publishing them.

In [ ]:
from fastmcp import FastMCP

# Define a tiny experimental tool in-process.
app = FastMCP("experimental")

@app.tool()
def greet(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}!"

# Test it via the in-memory transport (no network).
async with Client(app) as client:
    result = await client.call_tool("greet", {"name": "Atlas"})
    print(result)